# 03 - Dense Embedding Models (Hue Foods RAG MVP)

Notebook này minh họa luồng xử lý của Phase 3 trong Hue Foods RAG:
- Dense embedding bằng local model `intfloat/multilingual-e5-small` trên CPU;
- Phân biệt tiền tố `passage: ` và `query: `;
- Tính toán Cosine Similarity trực tiếp trên các vector chuẩn hóa.

Toàn bộ quá trình chạy hoàn toàn local, không cần OpenRouter API key hay kết nối mạng bên ngoài.


In [ ]:
import math
import sys
import time
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError("Không tìm thấy thư mục backend/.")

from core.settings_loader import load_settings
from embedding.embedder import E5Embedder
from ingestion.chunking.markdown_chunker import chunk_foods_markdown

settings = load_settings()
embedding_cfg = settings["embedding"]
print("model:", embedding_cfg["model"])
print("vector_size:", embedding_cfg["vector_size"])
print("device:", embedding_cfg["device"])
print("batch_size:", embedding_cfg["batch_size"])


## Dense Embedding với Multilingual E5

Mô hình E5 yêu cầu phân biệt rõ hai vai trò:
- Tài liệu (documents/passages) được gắn tiền tố `passage: `;
- Câu truy vấn (queries) được gắn tiền tố `query: `.

`E5Embedder` tự động xử lý các tiền tố này và gọi trực tiếp `SentenceTransformer.encode()` với `batch_size=64` và `normalize_embeddings=True`.


In [ ]:
embedder = E5Embedder(
    model_id=embedding_cfg["model"],
    dimension=embedding_cfg["vector_size"],
    device=embedding_cfg["device"],
    batch_size=embedding_cfg["batch_size"],
)

chunks = chunk_foods_markdown()
texts = [chunk["text"] for chunk in chunks]

started = time.perf_counter()
dense_vectors = embedder.embed_documents(texts)
elapsed_seconds = time.perf_counter() - started


In [ ]:
first_vector = dense_vectors[0]
first_norm = math.sqrt(sum(v * v for v in first_vector))

print("model_id:", embedder.model_id)
print("chunk_count:", len(chunks))
print("dense_shape:", f"{len(dense_vectors)} x {len(first_vector)}")
print("first_vector_norm:", round(first_norm, 4))
print("elapsed_seconds:", round(elapsed_seconds, 2))


### Ý nghĩa của kết quả Dense Embedding

- `dense_shape`: 572 vectors x 384 dimensions tương ứng với toàn bộ 572 canonical chunks của Hue Foods RAG.
- `first_vector_norm`: Chuẩn L2 xấp xỉ 1.0 xác nhận các vector đã được chuẩn hóa để tính Cosine Similarity bằng tích vô hướng (dot product).
- `elapsed_seconds`: Thời gian chạy thực tế trên CPU (dùng để quan sát hiệu năng, không phải điều kiện cứng).


In [ ]:
sample = "Bún bò Huế"
query_vector = embedder.embed_query(sample)
document_vector = embedder.embed_documents([sample])[0]

cosine_similarity = sum(
    q * d for q, d in zip(query_vector, document_vector)
)
print("sample text:", repr(sample))
print("query vector dimension:", len(query_vector))
print("document vector dimension:", len(document_vector))
print("query/document cosine:", round(cosine_similarity, 4))


## Handoff sang Phase 8 (Model Selection)

Các mô hình embedding từ xa (như OpenRouter Qwen3 Embedding) sẽ được đánh giá và so sánh toàn diện với baseline E5 trong Phase 8 trên cùng tập dữ liệu chuẩn và các metric đo lường thống nhất.
